In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
DATA = Path("/content/drive/MyDrive/Olist/data")          # CSV를 넣어둔 폴더 경로 (본인 환경에 맞게 수정)
HB = "beleza_saude"          # health_beauty의 원천(포르투갈어) 카테고리 값


def load(name):
    return pd.read_csv(DATA / f"{name}.csv")


customers = load("olist_customers_dataset")
orders    = load("olist_orders_dataset")
items     = load("olist_order_items_dataset")
payments  = load("olist_order_payments_dataset")
products  = load("olist_products_dataset")
sellers   = load("olist_sellers_dataset")
trans     = load("product_category_name_translation")

In [ ]:
# [1] 테이블별 행 수 -> README '원천 데이터 규모'에 사용
print("[1] 행 수")
for n, df in {"customers": customers, "orders": orders, "items": items,
              "payments": payments, "products": products, "sellers": sellers}.items():
    print(f"  {n:10s} {len(df):>8,}")

[1] 행 수
  customers    99,441
  orders       99,441
  items       112,650
  payments    103,886
  products     32,951
  sellers       3,095


In [ ]:
# [2] 회원 식별자 (D-011): customer_id는 주문마다 새로 발급되는지 확인
print("\n[2] 회원 식별자")
print("  customer_id 고유 수       :", customers.customer_id.nunique())
print("  customer_unique_id 고유 수:", customers.customer_unique_id.nunique())
multi = customers.groupby("customer_unique_id").customer_id.nunique()
print("  customer_id를 2개 이상 가진 사람 수:", (multi > 1).sum())


[2] 회원 식별자
  customer_id 고유 수       : 99441
  customer_unique_id 고유 수: 96096
  customer_id를 2개 이상 가진 사람 수: 2997


In [ ]:
# [3] 카테고리: 원천 값과 번역 확인, 카테고리 없는 상품 확인
print("\n[3] 카테고리")
print("  카테고리 NULL 상품 수:", products.product_category_name.isna().sum())
print("  beleza_saude 상품 수 :", (products.product_category_name == HB).sum())
print("  번역 확인:", trans[trans.product_category_name == HB].to_dict("records"))



[3] 카테고리
  카테고리 NULL 상품 수: 610
  beleza_saude 상품 수 : 2444
  번역 확인: [{'product_category_name': 'beleza_saude', 'product_category_name_english': 'health_beauty'}]


In [ ]:
# [4] 필터 기준 (D-012): health_beauty 주문에 다른 카테고리가 섞여 있는지
it = items.merge(products[["product_id", "product_category_name"]],
                 on="product_id", how="left")
it["is_hb"] = it.product_category_name == HB
per_order = it.groupby("order_id").is_hb.agg(["any", "all"])   # any: 하나라도 HB, all: 전부 HB
hb_order_ids = per_order[per_order["any"]].index
print("\n[4] 필터 기준")
print("  health_beauty 포함 주문 수     :", len(hb_order_ids))
print("  그중 다른 카테고리가 섞인 주문 :", (~per_order.loc[hb_order_ids, "all"]).sum())



[4] 필터 기준
  health_beauty 포함 주문 수     : 8836
  그중 다른 카테고리가 섞인 주문 : 72


In [ ]:
# [5] 수량 (D-013): 같은 주문에 같은 상품이 여러 행으로 들어가는지
hb_items = it[it.is_hb]
dup = hb_items.groupby(["order_id", "product_id"]).size()
print("\n[5] 수량")
print("  같은 주문·같은 상품이 2행 이상:", (dup > 1).sum(), "/", len(dup))
print("  최대 반복 행 수               :", dup.max())


[5] 수량
  같은 주문·같은 상품이 2행 이상: 461 / 9022
  최대 반복 행 수               : 10


In [ ]:
# [6] 복수 결제 (D-014): 한 주문에 결제가 여러 건인 경우
hb_pay = payments[payments.order_id.isin(hb_order_ids)]
n_pay = hb_pay.groupby("order_id").payment_sequential.max()
print("\n[6] 결제")
print("  결제 2건 이상인 주문 수:", (n_pay > 1).sum(), "/", len(n_pay))
print(hb_pay.payment_type.value_counts().to_string())


[6] 결제
  결제 2건 이상인 주문 수: 200 / 8835
payment_type
credit_card    6898
boleto         1704
voucher         369
debit_card      146


In [ ]:
# [7] 판매자 (D-009): health_beauty를 몇 명의 판매자가 팔았는지
print("\n[7] 판매자")
print("  health_beauty 판매자 수:", hb_items.seller_id.nunique())



[7] 판매자
  health_beauty 판매자 수: 492


In [ ]:
# [8] 기간·주문상태 (D-016, 주문상태 코드 매핑)
hb_ord = orders[orders.order_id.isin(hb_order_ids)]
ts = pd.to_datetime(hb_ord.order_purchase_timestamp)
print("\n[8] 기간·주문상태")
print("  구매일시 범위:", ts.min(), "~", ts.max())
print(hb_ord.order_status.value_counts().to_string())


[8] 기간·주문상태
  구매일시 범위: 2016-09-15 12:16:38 ~ 2018-08-29 14:18:28
order_status
delivered     8647
shipped        107
canceled        36
invoiced        32
processing      14


In [ ]:
# [9] 금액 정합성 미리보기: 주문 품목(가격+운임) 합계 vs 결제 합계
#     -> 나중에 품질 점검 SQL로 다시 하게 될 점검의 사전 확인
all_items = items[items.order_id.isin(hb_order_ids)]
item_sum = (all_items.price + all_items.freight_value).groupby(all_items.order_id).sum()
pay_sum = hb_pay.groupby("order_id").payment_value.sum()
diff = (item_sum - pay_sum).abs().dropna()
print("\n[9] 금액 정합성")
print("  품목 합계와 결제 합계가 0.01 이상 차이 나는 주문:", (diff > 0.01).sum(), "/", len(diff))
print("  결제 기록이 없는 주문:", len(set(hb_order_ids) - set(pay_sum.index)))



[9] 금액 정합성
  품목 합계와 결제 합계가 0.01 이상 차이 나는 주문: 17 / 8835
  결제 기록이 없는 주문: 1


In [16]:
# 한 상품을 여러 판매자가 판 경우가 얼마나 있는지 -> 판매자가 상품 속성이 아닌 거래 속성임을 확인
sellers_per_product = hb_items.groupby("product_id").seller_id.nunique()
print("2명 이상의 판매자가 판 상품 수:", (sellers_per_product > 1).sum(), "/", len(sellers_per_product))

2명 이상의 판매자가 판 상품 수: 147 / 2444


In [17]:
# 같은 주문·같은 상품 안에서 가격이 서로 다른 경우가 있는지
price_var = hb_items.groupby(["order_id", "product_id"]).price.nunique()
print("가격이 다른 반복 행 그룹 수:", (price_var > 1).sum())

가격이 다른 반복 행 그룹 수: 0
